# 📥 safetensors 探索と読み込み量

model-mtp を target も読むか、実 read 量の見積り。


In [ ]:
import glob
import pathlib
import subprocess
import sys
import zipfile

subprocess.run([sys.executable, "-m", "pip", "download", "--no-cache-dir",
                "--no-deps", "vllm==0.32.0", "-d", "/tmp/w"],
               capture_output=True, text=True)
whl = glob.glob("/tmp/w/*.whl")
print("wheel:", whl)
if not whl:
    raise SystemExit("wheel 取得失敗")
zipfile.ZipFile(whl[0]).extractall("/tmp/vl")
root = pathlib.Path("/tmp/vl/vllm")
print("py files:", len(list(root.rglob("*.py"))))

def find_root():
    import glob
    slug = "qwen38-27b-uncensored-hauhaucs-awq-mtp"
    ordered = []
    for pat in ("/kaggle/input/*/config.json", "/kaggle/input/*/*/config.json",
                "/kaggle/input/*/*/*/config.json"):
        ordered += glob.glob(pat)
    cands = ([c for c in ordered if slug in c]
             or [c for c in ordered if "qwen" in c.lower()]
             or ordered)
    print("config.json 候補:", cands[:3])
    if not cands:
        print("/kaggle/input 直下:", glob.glob("/kaggle/input/*")[:10])
        raise SystemExit("dataset が見つからない")
    return cands[0].rsplit("/", 1)[0]

import json
import os

ROOT = find_root()

print("=== Dataset 内の safetensors ===")
for fn in sorted(os.listdir(ROOT)):
    if fn.endswith((".safetensors", ".bin", ".pt")):
        p = os.path.join(ROOT, fn)
        print(f"  {fn}: {os.path.getsize(p) / 2**30:.2f} GiB")

print("\n=== default_loader の探索パターン ===")
t = (root / "model_executor/model_loader/default_loader.py").read_text(errors="replace")
for i, l in enumerate(t.splitlines(), 1):
    s = l.strip()
    if "allow_patterns" in s and ("=" in s or "glob" in s):
        print(f"  {i}: {s[:150]}")

print("\n=== index 無しのときの扱い ===")
w = (root / "model_executor/model_loader/weight_utils.py").read_text(errors="replace")
i = w.find("def filter_duplicate_safetensors_files")
print(w[i:i + 1400])

print("\n=== target と draft で同じファイルを読むか ===")
print("  - target: allow_patterns=['*.safetensors'] → model.safetensors と")
print("            model-mtp.safetensors の両方を glob する（index.json 無しのため絞れない）")
print("  - target 側 mapper は 'mtp.' -> None なので mtp の 15 テンソルは破棄")
print("  - draft: 同ディレクトリを再度読む → 実質 0.79 GiB を 2 回読む")

print("\n=== 読み込み量と時間（実測 82 MiB/s）===")
bw = 82 * 2**20
main = os.path.getsize(ROOT + "/model.safetensors")
mtp = os.path.getsize(ROOT + "/model-mtp.safetensors")
print(f"  main={main / 2**30:.2f} GiB mtp={mtp / 2**30:.2f} GiB")
for label, total in (("target(main+mtp)", main + mtp),
                     ("draft(mtp のみ?)", mtp),
                     ("合計(素直な見積り)", main + mtp * 2)):
    print(f"  {label:20s} {total / 2**30:5.2f} GiB → {total / bw:6.1f}s")
print(f"  TP=2 で各 GPU が半分読む場合の target: {(main + mtp) / 2 / bw:.1f}s")

print("\n=== mtp テンソルの shape（BF16 なので size からは分からない）===")
import struct


def header(path):
    with open(path, "rb") as f:
        ln = struct.unpack("<Q", f.read(8))[0]
        return json.loads(f.read(ln).decode("utf-8", "replace"))


h = header(ROOT + "/model-mtp.safetensors")
for k, v in sorted(h.items()):
    if k == "__metadata__":
        continue
    size = v["data_offsets"][1] - v["data_offsets"][0]
    print(f"  {k:44s} {v.get('dtype'):5s} {size / 2**20:8.2f} MiB")

print("\n=== config の ignore が mtp を含むか ===")
cfg = json.load(open(ROOT + "/config.json"))
ig = (cfg.get("quantization_config") or {}).get("ignore") or []
print("  ignore 件数:", len(ig), "| mtp を含む:", [x for x in ig if "mtp" in x][:3])
